# RQ1 · Country-year publication panel
### For a chosen biomedical field, which country characteristics predict publication output, and how has the geographic distribution shifted over time?

**Design.**
- Unit of analysis: country × year.
- Sample: all countries with ≥ 1 article/year in the field, 2000-2023.
- Field: CRISPR (`"CRISPR-Cas Systems"[mesh] OR "CRISPR"[tiab]`)  -  swap
  this for any MeSH area you care about (single-cell, mRNA vaccines, etc.).
- Outcome: `log(1 + n_articles_country_year)`.
- Regressors: `log(GDP per capita)`, `log(population)`, `log(1 + prior-year output)` (lagged dependent, to capture persistence).
- Fixed effects: country + year.
- SEs: clustered at country.

**Why lagged dependent?** Research output is highly persistent (labs stay
productive year over year). Without a lag term the country-FE estimates
would soak up autocorrelation and mis-attribute persistence to level
differences. Including it turns this into a partial-adjustment model.

**Compute footprint.** PubMed pull ≈ 50K-100K records depending on the
MeSH term. Local pull with an API key takes ~15-30 minutes; OSG cuts
that to a few minutes and lets us do robustness runs with 1,000 bootstrap
draws.


In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.simplefilter("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.ncbi_client import pubmed_query_to_dataframe
from src.regression import fixed_effects_panel


## 1. Pull the literature panel

In [ ]:
FIELD_QUERY = '"CRISPR-Cas Systems"[mesh] OR "CRISPR"[tiab]'
YEARS = (2000, 2023)

lit = pubmed_query_to_dataframe(
    FIELD_QUERY,
    mindate=f"{YEARS[0]}/01/01",
    maxdate=f"{YEARS[1]}/12/31",
    max_records=100000,   # generous cap; adjust per query
)
print(f"{len(lit):,} articles pulled  |  {lit['country'].notna().sum():,} with country tag")
lit.head()


## 2. Aggregate to country × year

In [ ]:
ct = (
    lit.dropna(subset=["year", "country"])
    .groupby(["country", "year"])
    .size().reset_index(name="n_articles")
)
ct["log1p_n"] = np.log1p(ct["n_articles"])
# Lagged output within country.
ct = ct.sort_values(["country", "year"])
ct["lag1_log1p_n"] = ct.groupby("country")["log1p_n"].shift(1)

print("Country-year cells:", len(ct))
ct.head()


## 3. Merge country economic covariates

We reuse the World Bank client from the sister repo if you have it
installed alongside; otherwise we pull the two indicators inline.


In [ ]:
import requests

def wb_pull(indicator, start, end):
    rows, page = [], 1
    while True:
        url = (f"https://api.worldbank.org/v2/country/all/indicator/{indicator}"
               f"?format=json&date={start}:{end}&per_page=20000&page={page}")
        meta, chunk = requests.get(url, timeout=60).json()
        rows.extend(chunk or [])
        if page >= meta.get("pages", 1): break
        page += 1
    return pd.DataFrame([
        {"iso3": r.get("countryiso3code"), "country_wb": r["country"]["value"],
         "year": int(r["date"]), indicator: r["value"]}
        for r in rows
    ]).dropna(subset=["iso3"]).query("iso3 != ''")

gdp = wb_pull("NY.GDP.PCAP.KD", YEARS[0]-1, YEARS[1])
pop = wb_pull("SP.POP.TOTL",    YEARS[0]-1, YEARS[1])
wb = gdp.merge(pop, on=["iso3","country_wb","year"])
wb["log_gdp_pc"] = np.log(wb["NY.GDP.PCAP.KD"].replace(0, np.nan))
wb["log_pop"]    = np.log(wb["SP.POP.TOTL"].replace(0, np.nan))
wb.head()


In [ ]:
# Crude name-to-iso3 crosswalk for common cases (extend as needed).
NAME_TO_ISO3 = {
    "USA": "USA", "United Kingdom": "GBR", "China": "CHN", "Japan": "JPN",
    "Germany": "DEU", "France": "FRA", "Italy": "ITA", "Spain": "ESP",
    "Canada": "CAN", "Australia": "AUS", "India": "IND", "Brazil": "BRA",
    "Netherlands": "NLD", "Sweden": "SWE", "Switzerland": "CHE",
    "South Korea": "KOR", "Israel": "ISR", "Denmark": "DNK", "Belgium": "BEL",
    "Singapore": "SGP", "Taiwan": "TWN", "Norway": "NOR", "Finland": "FIN",
    "Poland": "POL", "Russia": "RUS", "Portugal": "PRT", "Ireland": "IRL",
    "Turkey": "TUR", "Iran": "IRN", "Saudi Arabia": "SAU", "Mexico": "MEX",
    "South Africa": "ZAF", "New Zealand": "NZL", "Argentina": "ARG",
    "Austria": "AUT", "Greece": "GRC", "Hong Kong": "HKG",
}
ct["iso3"] = ct["country"].map(NAME_TO_ISO3)
panel = ct.merge(wb[["iso3","year","log_gdp_pc","log_pop"]], on=["iso3","year"], how="left")
panel = panel.dropna(subset=["log_gdp_pc","log_pop","lag1_log1p_n"])
print("Regression panel:", panel.shape)


## 4. Visual  -  where the output is concentrated

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
top10 = ct.groupby("country")["n_articles"].sum().nlargest(10).index
for c in top10:
    sub = ct[ct["country"] == c]
    ax.plot(sub["year"], sub["n_articles"], marker="o", label=c)
ax.legend(fontsize=8, ncol=2); ax.set_ylabel("Articles/year"); ax.grid(alpha=0.3)
ax.set_title(f"Top-10 country trajectories, PubMed field query")
fig.tight_layout()


## 5. Two-way FE regression

In [ ]:
res = fixed_effects_panel(
    panel,
    dependent="log1p_n",
    regressors=["lag1_log1p_n", "log_gdp_pc", "log_pop"],
    entity="iso3",
    entity_effects=True, time_effects=True,
    name="RQ1  -  Publication output panel (country × year)",
)
print(res.as_markdown())


In [ ]:
from pathlib import Path
Path("../results").mkdir(exist_ok=True)
res.coef_table.to_csv("../results/rq1_country_publication_panel_coefs.csv")
